### Menyiapkan Environment

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
from datetime import datetime
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 22:28:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


### A. Menyiapkan Streaming Source

In [2]:
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

folder_stream = "tugas10_stream"
shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("df_stream.isStreaming:", df_stream.isStreaming)

df_stream.isStreaming: True


### B. Agregasi per Cabang

In [3]:
df_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang dimulai.")

Query agregasi_cabang dimulai.


### C. Agregasi Berbasis Window Waktu

In [4]:
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

df_total_waktu = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu = df_total_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu dimulai.")

Query agregasi_waktu dimulai.


### D. Menguji dengan Data Simulasi

In [5]:
waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P1", "P2", "P3"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Latte", "Americano", "Croissant"],
    "qty": [2, 1, 3],
    "harga_satuan": [28000, 22000, 18000],
    "waktu_pesan": [waktu] * 3,
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(8)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          110000|             2|
| Yogyakarta|           22000|             1|
+-----------+----------------+--------------+



In [7]:
# batch 2 dengan kombinasi cabang yang berbeda
waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P4", "P5", "P6"],
    "nama_cabang": ["Solo", "Yogyakarta", "Magelang"],
    "nama_menu": ["Latte", "Cappuccino", "Americano"],
    "qty": [1, 2, 2],
    "harga_satuan": [28000, 25000, 22000],
    "waktu_pesan": [waktu] * 3,
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(8)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Batch 2 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          154000|             3|
| Yogyakarta|           72000|             2|
|       Solo|           28000|             1|
+-----------+----------------+--------------+



In [8]:
# batch 3 dengan cabang Semarang
waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P7", "P8", "P9"],
    "nama_cabang": ["Solo", "Magelang", "Semarang"],
    "nama_menu": ["Cappuccino", "Latte", "Croissant"],
    "qty": [3, 1, 4],
    "harga_satuan": [25000, 28000, 18000],
    "waktu_pesan": [waktu] * 3,
}).to_csv(f"{folder_stream}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(8)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Batch 3 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          182000|             4|
|       Solo|          103000|             2|
| Yogyakarta|           72000|             2|
|   Semarang|           72000|             1|
+-----------+----------------+--------------+



In [9]:
# hasil agregasi_waktu sebagai bukti query window berjalan
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_pesanan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

+-------------------+-------------------+----------------+--------------+
|start              |end                |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+----------------+--------------+
|2026-11-06 22:58:00|2026-11-06 23:00:00|132000          |3             |
|2026-11-06 23:34:00|2026-11-06 23:36:00|122000          |3             |
|2026-11-06 23:36:00|2026-11-06 23:38:00|175000          |3             |
+-------------------+-------------------+----------------+--------------+



### E. Sink Berbasis Berkas

In [10]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query file sink dimulai.")

Query file sink dimulai.


In [11]:
# kirim satu batch baru
waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P10", "P11", "P12"],
    "nama_cabang": ["Semarang", "Magelang", "Solo"],
    "nama_menu": ["Americano", "Cappuccino", "Croissant"],
    "qty": [2, 1, 2],
    "harga_satuan": [22000, 25000, 18000],
    "waktu_pesan": [waktu] * 3,
}).to_csv(f"{folder_stream}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark memproses...")
time.sleep(8)

Batch 4 dikirim. Menunggu Spark memproses...


In [12]:
!ls tugas10_output

part-00000-0b7702d4-71b4-46dd-a08e-945e57af69ff-c000.snappy.parquet
part-00000-2e1b0f12-7495-4bbb-a40c-ec02621fb0b6-c000.snappy.parquet
part-00001-884a146e-7850-4f5b-91de-0097d366fa9b-c000.snappy.parquet
part-00002-72b97afd-a3ad-4150-ad8f-f91418b1c0ea-c000.snappy.parquet
_spark_metadata


In [13]:
# membaca kembali berkas
df_hasil = spark.read.parquet("tugas10_output")
df_hasil.show()
print("Jumlah baris:", df_hasil.count())

+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|       P10|   Semarang| Americano|  2|       22000|2026-11-06 23:41:38|
|       P11|   Magelang|Cappuccino|  1|       25000|2026-11-06 23:41:38|
|       P12|       Solo| Croissant|  2|       18000|2026-11-06 23:41:38|
|        P4|       Solo|     Latte|  1|       28000|2026-11-06 23:35:27|
|        P5| Yogyakarta|Cappuccino|  2|       25000|2026-11-06 23:35:27|
|        P6|   Magelang| Americano|  2|       22000|2026-11-06 23:35:27|
|        P1|   Magelang|     Latte|  2|       28000|2026-11-06 22:58:15|
|        P2| Yogyakarta| Americano|  1|       22000|2026-11-06 22:58:15|
|        P3|   Magelang| Croissant|  3|       18000|2026-11-06 22:58:15|
|        P7|       Solo|Cappuccino|  3|       25000|2026-11-06 23:37:49|
|        P8|   Magelang|     Latte|  1|       28000

### Tutup semua query dan SparkSession

In [14]:
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.
